In [ ]:
# Optotagging analysis: read a Phy-curated Kilosort 4 sorting (run on a
# concatenated behavior+optotagging recording, see notebook 2.1), split spikes
# back per session using session_boundaries.json, then classify units as
# opto-tagged using the laser events from the optotagging session.

In [ ]:
from pathlib import Path
import yaml

# ======== parameters to change ========
session_name    = "IM-1979_2026-09-04"
base_dir_output = Path(r"E:/")
tagging_probe = "probeB"
kilosort_dir  = "kilosort4"                 # KS4 results subfolder name under <processed_session>/<tagging_probe>/
unit_criteria = {"quality": ["good"]}       # column -> criterion; see src.optotagging.load_curated_sorting
overwrite_summary = False                   # False = reuse optotagging_summary.csv if it already exists

opto_figures_mode = "plot"                  # "save" = all tagged units -> one PDF, no inline plots
                                            # "plot" = plot n_plot_preview random tagged units inline
n_plot_preview = 4
# ======================================

metadata_dir  = Path.cwd().parent
metadata_path = metadata_dir / "metadata" / f"{session_name}.yaml"
with open(metadata_path) as f:
    config = yaml.safe_load(f)

processed_path = base_dir_output / config["animal"] / "ephys" / config["sessions"]["processed"]
probe_names       = config["streams"]["probes"]

# The one probe to load + analyze for optotagging (must be Phy-curated already)
if tagging_probe not in probe_names:
    raise ValueError(f"tagging_probe={tagging_probe!r} not in this session's probes {probe_names}")

print(f"processed_session = {processed_path}")
print(f"tagging_probe     = {tagging_probe}  (available: {probe_names})")

In [ ]:
# ── load curated sorting for tagging_probe ──────────────────────────────────
import sys, json

sys.path.append('..')
from src import load_curated_sorting

sorting = load_curated_sorting(processed_path, tagging_probe, kilosort_dir=kilosort_dir,
                               unit_criteria=unit_criteria)

print(f"{tagging_probe}: {len(sorting.unit_ids)} units kept (unit_criteria={unit_criteria})")
display(sorting)

In [ ]:
# ── SANITY CHECK ONLY (not used by the optotagging analysis below) ──────────
# Split the concatenated sorting back per session via session_boundaries.json
# and check per-session firing rates look sane — confirms the sorting loaded
# and the concat-axis coordinate maths are right before we analyze anything.
import pandas as pd
from src import split_sorting_by_session

with open(processed_path / "session_boundaries.json") as f:
    session_boundaries = json.load(f)
sessions = session_boundaries["streams"][tagging_probe]["sessions"]
sorting_by_session = split_sorting_by_session(sorting, sessions)

fs = sorting.get_sampling_frequency()
quality_by_unit = dict(zip(sorting.unit_ids, sorting.get_property("quality")))

rows = []
for session in sessions:
    sid = session["session_id"]
    duration_s = (session["sample_end"] - session["sample_start"]) / fs
    for unit_id in sorting_by_session[sid].unit_ids:
        n_spikes = len(sorting_by_session[sid].get_unit_spike_train(unit_id))
        rows.append({"unit_id": unit_id, "session_id": sid,
                     "quality": quality_by_unit[unit_id], "n_spikes": n_spikes,
                     "firing_rate_hz": n_spikes / duration_s})
pd.DataFrame(rows).pivot(index=["unit_id", "quality"], columns="session_id",
                          values="firing_rate_hz")

In [ ]:
# ── load laser events (+ per-block power / pulse width from the protocol csv) ──
import pandas as pd
from src import load_laser_events, load_laser_protocol, annotate_laser_events_with_protocol

# NOTE: use the unsliced `sorting` (concatenated-axis sample indices), not
# sorting_by_session["optotagging"] — laser_events.csv's on_sample_concat is
# in that same concatenated-axis coordinate system (see src.optotagging
# module docstring / load_laser_events). Onsets are lag-corrected (TTL + the
# ADC->probe artifact lag estimated in notebook 2.1).
laser_events = load_laser_events(processed_path, probe_name=tagging_probe, session_id="optotagging")

# pulse-count mismatch vs the protocol raises; pulse width / rate mismatch warns
protocol = load_laser_protocol(processed_path / "laser_control_protocol_dual_mega2560.csv")
fs = sorting.get_sampling_frequency()
laser_events = annotate_laser_events_with_protocol(laser_events, protocol, fs=fs)
print(laser_events.groupby(["color", "block", "power", "pulse_width_ms", "protocol_frequency_hz"]).size())

# laser-offset mask length used by notebook 2.1 -> keeps the SALT baseline clear of it
with open(processed_path / "preprocessing.json") as f:
    preprocessing = json.load(f)
post_off_ms = preprocessing["probes"][tagging_probe]["mask_laser_artifact"]["post_off_ms"]

# overrides of src.optotagging.DEFAULT_OPTOTAG_PARAMS (defaults = Dan's opto_1hz_salt.json)
opto_params = {"post_off_ms": post_off_ms}
print(f"\npost_off_ms = {post_off_ms}")

In [ ]:
# ── waveform check: reference vs laser-evoked mean waveform (Pearson r) ───────
# Reads the binary Kilosort ran on (params.py dat_path), opened read-only.
from src import read_kilosort_params, compute_waveform_correlation

kilosort_path = processed_path / tagging_probe / kilosort_dir
ks_params = read_kilosort_params(kilosort_path)
print(f"dat: {ks_params['dat_path']}  ({ks_params['n_channels_dat']} channels)")

wave_df = compute_waveform_correlation(sorting, laser_events, ks_params["dat_path"],
                                       ks_params["n_channels_dat"], params=opto_params)
print(wave_df.groupby("color").agg(n_units=("unit_id", "size"), n_wave_pass=("wave_pass", "sum"),
                                   median_r=("wave_corr", "median")))

In [ ]:
# ── per (unit, color, power) laser response + tagging ─────────────────────────
# opto_df: one row per (unit, color, power) — each power level tested separately.
# `is_tagged` = SALT p < alpha & reliability/n_responded pass & waveform r >= min
# for that colour; `color_tagged` = tagged for that colour at >= 1 of its powers.
from src import compute_population_opto_response

summary_path = processed_path / tagging_probe / "optotagging_summary.csv"
if summary_path.exists() and not overwrite_summary:
    print(f"reusing existing {summary_path.name} (set overwrite_summary=True to recompute)")
    opto_df = pd.read_csv(summary_path).sort_values("unit_id", kind="stable").reset_index(drop=True)
else:
    opto_df = compute_population_opto_response(sorting, laser_events, params=opto_params, wave_df=wave_df)
    opto_df.to_csv(summary_path, index=False)
    print(f"\nwrote {summary_path}")

color_tagged = opto_df[["unit_id", "color", "color_tagged"]].drop_duplicates()
print(f"{opto_df['is_tagged'].sum()} / {len(opto_df)} (unit, color, power) groups tagged")
print(f"{color_tagged['color_tagged'].sum()} (unit, color) tagged at >= 1 power level")
opto_df[opto_df["is_tagged"]]                      # sorted by unit_id

In [ ]:
# ── population overview: reliability vs latency, tagged units highlighted ────
%matplotlib inline
from src import plot_opto_response, plot_opto_population

plot_opto_population(opto_df)

In [ ]:
# ── per-unit raster + PSTH for the tagged units ────────────────────────────
#   opto_figures_mode == "plot" : plot n_plot_preview random tagged units inline
#   opto_figures_mode == "save" : save every tagged unit to disk, format set by figure_format
#       figure_format == "pdf" : all units -> one multi-page optotagging_figures.pdf
#       figure_format == "svg" / "png" : one file per unit, in optotagging_figures_<format>/
#                                         (png saved with a transparent background, 300 dpi)
import random
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm

opto_figures_mode = "save"  # "plot" or "save"
figure_format = "pdf"       # "pdf", "svg", or "png" -- only used when opto_figures_mode == "save"
drop_masked_spikes = False  # True = hide spikes inside laser artifact-mask windows from rasters/PSTHs
                            # (display only, warns per unit; tagging results are unaffected)
plot_kwargs = dict(params=opto_params, drop_masked_spikes=drop_masked_spikes,
                   mask_params=preprocessing["probes"][tagging_probe]["mask_laser_artifact"])

tagged_unit_ids = sorted(opto_df.loc[opto_df["is_tagged"], "unit_id"].unique().tolist())   # ascending unit_id
print(f"{len(tagged_unit_ids)} tagged units")

if opto_figures_mode == "plot":
    for unit_id in sorted(random.sample(tagged_unit_ids, k=min(n_plot_preview, len(tagged_unit_ids)))):
        plot_opto_response(sorting, laser_events, unit_id, **plot_kwargs)

elif opto_figures_mode == "save":
    if figure_format == "pdf":
        pdf_path = processed_path / tagging_probe / "optotagging_figures.pdf"
        with PdfPages(pdf_path) as pdf:
            for unit_id in tqdm(tagged_unit_ids, desc="pdf pages", unit="unit"):
                fig = plot_opto_response(sorting, laser_events, unit_id, **plot_kwargs)
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
        print(f"saved {len(tagged_unit_ids)} pages to {pdf_path}")

    elif figure_format in ("svg", "png"):
        out_dir = processed_path / tagging_probe / f"optotagging_figures_{figure_format}"
        out_dir.mkdir(exist_ok=True)
        for unit_id in tqdm(tagged_unit_ids, desc=f"{figure_format} files", unit="unit"):
            fig = plot_opto_response(sorting, laser_events, unit_id, **plot_kwargs)
            fig_path = out_dir / f"unit_{unit_id}.{figure_format}"
            if figure_format == "png":
                fig.savefig(fig_path, bbox_inches="tight", transparent=True, dpi=300)
            else:
                fig.savefig(fig_path, bbox_inches="tight", transparent=True)
            plt.close(fig)
        print(f"saved {len(tagged_unit_ids)} files to {out_dir}")

    else:
        raise ValueError(f"figure_format must be 'pdf', 'svg', or 'png', got {figure_format!r}")

else:
    raise ValueError(f"opto_figures_mode must be 'plot' or 'save', got {opto_figures_mode!r}")